In [1]:
import pyterrier as pt
from pyterrier_dr import TctColBert, FlexIndex
from pyterrier_adaptive import GAR
from typing import Optional
import numpy as np
from collections import Counter
import pyterrier as pt
import pandas as pd
import ir_datasets
# Typing imports for type annotations
from pyterrier.model import add_ranks
import torch
from torch.nn import functional as F
from transformers import T5Config, T5Tokenizer, T5ForConditionalGeneration
from pyterrier.transformer import TransformerBase
import re
import scipy.sparse
import torch
import scipy
from pyterrier_t5 import MonoT5ReRanker
import json
from pyterrier_pisa import PisaIndex

from src.utils import *
from src.lightningmodule import *
from src.GNN import *
import warnings
logger = ir_datasets.log.easy()
if not pt.started(): 
    pt.init()
set_determinism_the_old_way(deterministic = True)


/home/peppe/anaconda3/envs/GNRR2/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/peppe/anaconda3/envs/GNRR2/lib/python3.8/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/home/peppe/anaconda3/envs/GNRR2/lib/python3.8/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
PyTerrier 0.10.0 has loaded Terrier 5.9 (built by craigm on 2024-05-02 17:40) and terrier-helper 0.0.8

No etc/terrier.properties, using terrier.default.properties for bootstrap configuration.


In [2]:
dataset_id = "msmarco_data"
embedding_name = 'tctcolbert'
K = 8
flex_index = FlexIndex(index_path=f'./data/msmarco-index_{embedding_name}/')

# Generate the corpus graph using the corpus_graph method of FlexIndex.
graph = flex_index.corpus_graph(k=K)

In [3]:
# bm25 = PisaIndex.from_dataset('msmarco_passage').bm25()
bm25 =  pt.BatchRetrieve.from_dataset('msmarco_passage', 'terrier_stemmed', wmodel='BM25')
text_field = "text"

monoT5 = MonoT5ReRanker(text_field=text_field)
TCTC = TctColBert('castorini/tct_colbert-msmarco')
TCTC2 = TctColBert('castorini/tct_colbert-v2-hnp-msmarco')



/home/peppe/anaconda3/envs/GNRR2/lib/python3.8/site-packages/transformers/models/t5/tokenization_t5.py:240: FutureWarning: This tokenizer was incorrectly instantiated with a model max length of 512 which will be corrected in Transformers v5.
For now, this behavior is kept to avoid breaking backwards compatibility when padding/encoding with `truncation is True`.
- Be aware that you SHOULD NOT rely on t5-base automatically truncating your input to 512 when padding/encoding.
- If you want to encode/pad to sequences longer than 512 you can either instantiate this tokenizer with `model_max_length` or pass `max_length` when encoding/padding.
- To avoid this warning, please instantiate this tokenizer with `model_max_length` set to your preferred value.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. I

In [4]:
import torch
import pandas as pd
import copy
import torch
from torch.nn import functional as F
import numpy as np
import numpy as np
class GNRR_Scorer(TransformerBase):
    def __init__(self,
                 batch_size=16,
                 text_field='text',
                 fast = False,
                 qrels = None,
                 config = None,
                 verbose=True, get_data = False, dataset = None):
        self.verbose = verbose
        self.batch_size = batch_size
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model_name = 'castorini/tct_colbert-msmarco'
        self.encoder = TctColBert(self.model_name, device = self.device)
        self.text_field = text_field
        n = 768
        self.fast = fast
        self.get_data = get_data
        input_features = n if config.aggr != 'concat' else n * 2
        self.dataset_n = dataset
        self.qrels = qrels

        
        if config.modality == 'local':
            
            mod = GNN_LG(input_features, config, modality = config.modality, conv_type = config.conv_type, device = self.device)
        
        elif config.modality == 'multistage':
            mod = GNN_LG(input_features, config, modality = config.modality, conv_type = config.conv_type, device = self.device)

        elif config.modality == 'global':
            mod = GNN_LG(input_features, config, modality = config.modality, conv_type = config.conv_type, device = self.device)
        elif config.modality == 'single':

            if config.conv_type != 'mlp':
                mod = GNN_NR(input_features, config, output_dim = 1, device=self.device)
            else:
                mod = MLP(input_features, config.hidden_dim, output_dim = 1, n_layers=config.n_layers, device=self.device, dropout_prob=config.dropout_prob)
        
        
        if config.model_path != '':
            model_checkpoint = torch.load(f'{config.model_path}', map_location=self.device)
            mod.load_state_dict(model_checkpoint, strict=False)

        print(self.device)
        self.model = mod.to(self.device)
        
        self.config = config

    def __str__(self):
        return f"GNRR_Scorer({self.model_name})"

    def transform(self, run, corpus_graph, corpus_graph_payload = None):
        self.model.eval()
        with torch.no_grad():    
        # CHANGE
            if self.get_data:
                dataset_n = self.dataset_n
                q_id = run['qid'].iloc[0]

                path = f'./data/msmarco_data/test_graphs/tensors/qid_{str(q_id)}_tensors/'
                if os.path.exists(path):
                   print("Path exists")
                   return

            topk_documents_df = run.drop_duplicates(subset='docno')
            print(len(topk_documents_df))

            if len(topk_documents_df) < 1000:
                print(len(topk_documents_df))
                print("Less than 1000")
            
            if self.get_data:
                # Retrieve the relevance labels for the top-k documents
                merged_df = topk_documents_df.merge(self.qrels.loc[:, ['qid', 'docno', 'label']], on=['qid', 'docno'], how='left')
                merged_df = merged_df.fillna(-1)
            
            queries, texts = topk_documents_df['query'], topk_documents_df[self.text_field]

            docs = texts
            # print(query)
            query_enc = self.encoder.encode_queries(queries.iloc[0:1])[0]

            docno_to_index = {docno: idx for idx, docno in enumerate(topk_documents_df['docno'].unique())}


        
            index_to_docno = {docno_to_index[k]: k for k in docno_to_index}

            
            if not self.fast:
                doc_encs = self.encoder.encode_docs(docs)
            else:
                doc_encs = np.empty((len(topk_documents_df), query_enc.shape[0]), dtype=query_enc.dtype)
                
                try:
                    print("not SKIPPED")

                    for doc in range(len(topk_documents_df)):
                        docno = index_to_docno[doc]
                        real_id = corpus_graph_payload[0][docno]
                        doc_encs[doc] = corpus_graph_payload[1][real_id]
                except IndexError:
                    print("SKIPPED")
                    doc_encs = self.encoder.encode_docs(docs)

                

            if self.get_data:
                dataset_n = self.dataset_n
                q_id = run['qid'].iloc[0]
                path = f'./data/{dataset_n}/tensors/qid_{str(q_id)}_tensors/'

                os.makedirs(path, exist_ok=True)

                rels = torch.from_numpy(np.array(merged_df['label']))

                print("Rels have shape: ", rels.shape)
                torch.save(rels, f'{path}qrels_tensor_new.pt')

                path2 = f'./data/{dataset_n}/metrics_directory/'#
                os.makedirs(path2, exist_ok=True)
                with open(path2+f'{str(q_id)}_docno_to_index_docids_mapping.json', 'w') as f:
                    dict_mapping_doc_to_index = {str(k): [v, 0] for k, v in docno_to_index.items()}
                    json.dump(dict_mapping_doc_to_index, f)
                

            corpus_sb = generate_corpus_subgraph_induced_by_query(topk_documents_df = topk_documents_df, complete_corpus_graph = corpus_graph)


            assert len(docno_to_index) == len(index_to_docno)


            for idx in range(len(doc_encs)):
                assert index_to_docno[idx] == get_key_from_value(docno_to_index, idx)

            adj_matrix = build_adjacency_matrix(corpus_sb, docno_to_index)

            adj_matrix = adjacency_matrix_to_coo(adj_matrix)
            
            query_feat = (torch.from_numpy(query_enc).clone().unsqueeze(0))
            
            x = (torch.from_numpy(doc_encs).clone()) 
            
            if self.get_data:
                torch.save(adj_matrix, f'{path}adjacency_matrix.pt')
                
                print("Adj have shape: ", adj_matrix.shape)
            
                torch.save(query_feat, f'{path}query_tensor_new.pt')
                print("query_feat have shape: ", query_feat.shape)
                torch.save(x, f'{path}doc_feat_tensor_new.pt')
                print("Doc have shape: ", x.shape)
            
            query_feat = query_feat.unsqueeze(0).to(self.device)

            x = x.unsqueeze(0).to(self.device)
          
            A = adj_matrix.unsqueeze(0).to(self.device)

            out = compute_output(x, A, query_feat, self.model, self.config.aggr, self.config.conv_type)

        ordered_list = sorted(list(index_to_docno.keys()))
        topk_indices = torch.topk(out, k = out.shape[0]).indices
        data = {
            'qid': [str(run['qid'].iloc[0])]*len(ordered_list),
            'query': queries,
            'docno': [],
            'score': [], 
            'rank': []
        }

        for doc in ordered_list:

            data['docno'].append(index_to_docno[doc])
            data['score'].append(out[doc].item())
            data['rank'].append(topk_indices.tolist().index(doc))
            
        df = pd.DataFrame(data)
        df = df.sort_values(by='rank')
    
        
        return df
    
def generate_corpus_subgraph_induced_by_query(                                          
    topk_documents_df: pd.DataFrame,
    complete_corpus_graph
) -> Dict[str, List[str]]:
    """
    Constructs and refines a corpus subgraph, focusing on relationships within a document subset.

    Conceptual Steps:
    1. Define Nodes: Identify and set the documents of interest as nodes in our subgraph. This step
    uses the 'documents_df' to extract document numbers, which will serve as nodes.

    2. Draw Edges: For each node, retrieve potential connections (edges) from the complete corpus graph.
    This involves fetching neighbors for each document from the comprehensive graph structure.

    3. Filter Edges: Refine the connections by ensuring each node (document) only connects to other nodes
    (documents) within our subset. This filtering process removes edges that lead outside the
    specified subset, maintaining the subgraph's integrity.

    4. Construct Subgraph: Populate the subgraph with nodes and their valid, filtered connections. This
    results in a dictionary where each key is a document number, and its value is a list of neighbor
    document numbers—all within the subset (i.e., valid neighbours).

    Args:
    - documents_df (pd.DataFrame): DataFrame containing documents of interest, identified by 'docno'.
    - graph_reference (NpTopKCorpusGraph): The complete corpus graph for neighbor retrieval.

    Returns:
    - Dict[str, List[str]]: Represents the corpus subgraph. Keys are document numbers ('docno'),
    and values are lists of neighbor document numbers, ensuring all are within the specified subset.
    """

    # Step 1: Define Nodes
    # Extract a set of document numbers to serve as valid nodes within our subgraph.
    valid_docnos = set(topk_documents_df['docno'])

    # Initialize the subgraph
    corpus_subgraph = {}
    found = 0
    for docno in valid_docnos:
        # Step 2: Draw Edges
        # Retrieve neighbors for the current document from the complete corpus graph.
        # CHANGE
        try:
            all_neighbors = complete_corpus_graph.neighbours(docno)
            found += 1
        except LookupError:
            warnings.warn(f"Document {docno} not found in the corpus graph.")
            continue
        # Step 3: Filter Edges
        # Filter these neighbors to include only those also present in our subset (valid_docnos).
        valid_neighbors = [neighbor for neighbor in all_neighbors if neighbor in valid_docnos]

        # Step 4: Construct Subgraph
        # Update our subgraph to include the current document and its filtered neighbors.
        corpus_subgraph[docno] = valid_neighbors  # Populate subgraph
    
    return corpus_subgraph
    
def build_adjacency_matrix(subgraph: Dict[str, list], docno_to_index: Dict[str, int]) -> np.ndarray:
    """
    Generates an adjacency matrix from a subgraph and a mapping of document numbers to indices.

    Parameters:
    subgraph (Dict[str, list]): A dictionary representing the subgraph with document numbers as keys.
    docno_to_index (Dict[str, int]): A dictionary mapping document numbers to their respective indices.

    Returns:
    np.ndarray: A symmetric adjacency matrix representing the graph.
    """
    # Error handling: Check if inputs are dictionaries
    if not isinstance(subgraph, dict) or not isinstance(docno_to_index, dict):
        raise ValueError("Both subgraph and docno_to_index must be dictionaries.")

    # Determine the size of the adjacency matrix
    # CHANGE
    matrix_size = len(docno_to_index)
    
    adjacency_matrix = np.zeros((matrix_size, matrix_size), dtype=int)

    # Iterate over each document and its neighbors in the subgraph
    for doc, neighbors in subgraph.items():
        if doc not in docno_to_index:
            raise KeyError(f"Document number {doc} not found in docno_to_index mapping.")
        doc_index = docno_to_index[doc]

        for neighbor in neighbors:
            if neighbor not in docno_to_index:
                raise KeyError(f"Neighbor {neighbor} of document {doc} not found in docno_to_index mapping.")
            
            neighbor_index = docno_to_index[neighbor]

            # Mark the connection in the matrix, ensuring symmetry
            adjacency_matrix[doc_index, neighbor_index] = adjacency_matrix[neighbor_index, doc_index] = 1

    return adjacency_matrix
    
    
def adjacency_matrix_to_coo(adjacency_matrix: np.ndarray) -> torch.Tensor:
    """
    Converts an adjacency matrix to COO format using PyTorch Geometric.

    Parameters:
    adjacency_matrix (np.ndarray): The adjacency matrix to be converted.

    Returns:
    torch.Tensor: Edge index tensor in COO format.
    """
    # Convert the numpy adjacency matrix to a SciPy sparse matrix (COO format)
    scipy_sparse_matrix = scipy.sparse.coo_matrix(adjacency_matrix)

    # Convert the SciPy sparse matrix to PyTorch Geometric COO format
    edge_index, edge_weight = from_scipy_sparse_matrix(scipy_sparse_matrix)

    return edge_index

In [5]:
class GNRR(pt.Transformer):
    """
    A transformer that implements the Graph-based Adaptive Re-ranker algorithm from
    MacAvaney et al. "Adaptive Re-Ranking with a Corpus Graph" CIKM 2022.

    Required input columns: ['qid', 'query', 'docno', 'score', 'rank']
    Output columns: ['qid', 'query', 'docno', 'score', 'rank', 'iteration']
    where iteration defines the batch number which identified the document. Specifically
    even=initial retrieval   odd=corpus graph    -1=backfilled
    
    """
    def __init__(self,
        scorer: pt.Transformer,
        corpus_graph: 'CorpusGraph',
        flex_index,
        text_field = 'abstract'):
        self.scorer = scorer
        self.corpus_graph = corpus_graph
        self.flex_index = flex_index
        self.text_field = text_field

        self.dataset_retr = pt.get_dataset('irds:msmarco-passage')
        
       

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Applies Graph-based Adaptive Re-ranking to the provided dataframe. Essentially,
        Algorithm 1 from the paper.
        """
        result = {'qid': [], 'query': [], 'docno': [], 'rank': [], 'score': []}

        result = pd.DataFrame(result)

        df = dict(iter(df.groupby(by=['qid'])))
        qids = df.keys()

        # RANKING ALGORITHM TO INCREASE THE RECALL
        
        payload = self.flex_index.payload()

        # RE-RANKING
        for i, qid in enumerate(qids):
            print(f"Currently processing: {i+1}/{len(qids)}")
            
            batch = df[qid].loc[:, ['qid', 'query', 'docno', 'score']]
                # go score the batch of document with the re-ranker

            add_texts = pt.text.get_text(self.dataset_retr, 'text')
            
            batch = add_texts(batch)

            # print(batch)
            inter_result = self.scorer.transform(batch, self.corpus_graph, corpus_graph_payload = payload)
            

            result = pd.concat([result, inter_result], axis=0, ignore_index=True)
       

            result['rank'] = result['rank'].astype(int)

        return result

In [6]:
class Config:
    def __init__(self, modality, conv_type, hidden_dim, dropout_prob, n_layers, aggr, model_path, heads = 1, n_layers_mlp = 0, lamb = None, K_multistage = 0, pooling_ratio = None, pooling = None, structure_learning = None, score = True, load = '', aggr_sage = 'mean', negatives = 1):
        self.modality = modality
        self.conv_type = conv_type
        self.hidden_dim = hidden_dim
        self.dropout_prob = dropout_prob
        self.n_layers = n_layers
        self.aggr = aggr
        self.heads = heads
        self.n_layers_mlp = n_layers_mlp
        self.model_path = model_path
        self.load = load
        self.score = score
        self.aggr_sage = aggr_sage
        self.negatives = negatives
        self.lamb = lamb
        self.pooling_ratio = pooling_ratio
        self.pooling = pooling
        self.structure_learning = structure_learning
        self.K_multistage = K_multistage


modal = 'local'
embedding_name = 'tctcolbert'

if modal == 'local' and embedding_name == 'tctcolbert':

    config_local_gcn = Config(modality='local', conv_type='gcn', hidden_dim=128, dropout_prob=0, score = True,
                    n_layers=2, n_layers_mlp = 2, aggr='hadamard', model_path=f'models/msmarco_data/gcn_8_True_tctcolbert (copy).pt')
    config_local_gcn_multi = Config(modality='multistage', conv_type='gcn', hidden_dim=128, dropout_prob=0.1, score = True,
                    n_layers=1, n_layers_mlp = 2, aggr='hadamard', K_multistage=100, model_path=f'models/msmarco_data/hadamard_1_789_0.01_0.0_128_0.1_gcn_multistage_2_tctcolbert2_100.pt')

    config_local_gat = Config(modality='local', conv_type='gat', hidden_dim=128, dropout_prob=0.3, score = True,
                    n_layers=1, n_layers_mlp = 1, aggr='hadamard', model_path=f'models/msmarco_data/gat/bestTrue_gat_hadamard_1_1_4_0.01_0_128_0.3_local_8_True_tctcolbert_1.pt', heads = 1)

    config_local_gin = Config(modality='local', conv_type='gin', hidden_dim=64, dropout_prob=0.1, score = True,
                    n_layers=2, n_layers_mlp = 1, aggr='hadamard', model_path=f'models/msmarco_data/gin/bestTrue_gin_hadamard_2_1_3_0.01_0_64_0.1_local_8_True_tctcolbert.pt')

    config_local_sage = Config(modality='local', conv_type='sage', hidden_dim=64, dropout_prob=0.1, score = True, aggr_sage = 'max',
                    n_layers=1, n_layers_mlp = 1, aggr='hadamard', model_path=f'models/msmarco_data/sage/bestTrue_sage_hadamard_1_1_3_0.01_0_64_0.1_local_8_True_tctcolbert_max.pt')

    config_local_signed = Config(modality='local', conv_type='signed', hidden_dim=64, dropout_prob=0.1, score = True,
                    n_layers=2, n_layers_mlp = 1, aggr='hadamard', model_path=f'models/msmarco_data/signed/bestTrue_signed_hadamard_2_1_2_0.01_0_64_0.1_local_8_True_tctcolbert_1.pt', negatives = 1)

    tct_scorer_local_gat = GNRR_Scorer(config = config_local_gat, text_field=text_field)
    tct_scorer_local_gcn = GNRR_Scorer(config = config_local_gcn, text_field=text_field)
    tct_scorer_local_gcn_multi = GNRR_Scorer(config = config_local_gcn_multi, text_field=text_field)

    tct_scorer_local_gin = GNRR_Scorer(config = config_local_gin, text_field=text_field)
    tct_scorer_local_sage = GNRR_Scorer(config = config_local_sage, text_field=text_field)
    tct_scorer_local_signed = GNRR_Scorer(config = config_local_signed, text_field=text_field)




elif modal == 'global' and embedding_name == 'tctcolbert':
    
    config_global_gcn_sl = Config(modality='global', conv_type='gcn', hidden_dim = 64, lamb = 0.8, dropout_prob = 0.1, n_layers = 1, n_layers_mlp = 1, pooling_ratio = 0.8, pooling = 'hierarchical', structure_learning = True, aggr='hadamard', model_path=f'models/msmarco_data/gcn_hierarchical_True_1.pt')

    config_global_gat_sl = Config(modality='global', conv_type='gat', hidden_dim = 64, lamb = 0.8, dropout_prob = 0.1, n_layers = 1, n_layers_mlp = 2, pooling_ratio = 0.8, pooling = 'hierarchical', structure_learning = True, aggr='hadamard', model_path=f'models/msmarco_data/gat_hierarchical_True_2.pt')

    tct_scorer_global_gcn_sl = GNRR_Scorer(config = config_global_gcn_sl, text_field=text_field)

    tct_scorer_global_gat_sl = GNRR_Scorer(config = config_global_gat_sl, text_field=text_field)




cuda
cuda
cuda
cuda
cuda
cuda


In [7]:
test = 't'

In [8]:
dataset = pt.get_dataset('irds:msmarco-passage/trec-dl-2019/judged')
dl19_2019 = dataset
filtered_get_topics = dataset.get_topics()
filtered_get_qrels = dataset.get_qrels()
from pyterrier.measures import * 


result = pt.Experiment(
  [ 
    bm25,
    #bm25 >> pt.text.get_text(pt.get_dataset('irds:msmarco-passage'), 'text') >> TCTC,
    bm25 >> GNRR(tct_scorer_local_gcn, graph, flex_index, text_field=text_field), 
    bm25 >> GNRR(tct_scorer_local_sage, graph, flex_index, text_field=text_field), 
    bm25 >> GNRR(tct_scorer_local_gat, graph, flex_index, text_field=text_field), 
    bm25 >> GNRR(tct_scorer_local_gin, graph, flex_index, text_field=text_field), 
    bm25 >> GNRR(tct_scorer_local_signed, graph, flex_index, text_field=text_field) 


  ],
  filtered_get_topics,
  filtered_get_qrels,
  baseline = 1,
  test = test,
  highlight = 'bold',
  names=['bm25', ' +GCN', ' +GraphSAGE',' +GAT', ' +GIN', ' +SignedConv'],#, 'MLP', 'MLP + GCN', 'MLP + GIN'],#, 'MLP', 'monoT5'],#, "TCTColbert+GCN", "TCTColbert+MLP+GCN", "TCTColbert+MLP", "MonoT5"],
  eval_metrics=[nDCG@10, P(rel = 2)@3, AP(rel=2), RR(rel = 2), R(rel=2)@1000]
)
print(result)
# result = pt.Experiment(
#   [ 
#     bm25,
#     bm25 >> pt.text.get_text(pt.get_dataset('irds:msmarco-passage'), 'text') >> TCTC, 
#     bm25 >> GNRR(tct_scorer_global_gcn_sl, graph, flex_index, text_field=text_field),
#     bm25 >> GNRR(tct_scorer_global_gat_sl, graph, flex_index, text_field=text_field)

#   ],
#   filtered_get_topics,
#   filtered_get_qrels,
#   baseline = 1,
#   names=['bm25', 'TCTColbert', ' +GCN_Global_SL', ' +GAT_Global_SL'],
#   test = 't',
#   highlight = 'bold',#, 'MLP', 'MLP + GCN', 'MLP + GIN'],#, 'MLP', 'monoT5'],#, "TCTColbert+GCN", "TCTColbert+MLP+GCN", "TCTColbert+MLP", "MonoT5"],
#   eval_metrics=[nDCG@10, P(rel = 2)@3, AP(rel=2), RR(rel = 2), R(rel=2)@1000] #[nDCG@10, nDCG@20, P(rel = 2)@10, P(rel = 2)@20, R(rel=2)@1000]
# )

Currently processing: 1/43
1000
Currently processing: 2/43
1000
Currently processing: 3/43
1000
Currently processing: 4/43
1000
Currently processing: 5/43
1000
Currently processing: 6/43
1000
Currently processing: 7/43
1000
Currently processing: 8/43
1000
Currently processing: 9/43
1000
Currently processing: 10/43
1000
Currently processing: 11/43
1000
Currently processing: 12/43
1000
Currently processing: 13/43
1000
Currently processing: 14/43
1000
Currently processing: 15/43
1000
Currently processing: 16/43
1000
Currently processing: 17/43
1000
Currently processing: 18/43
1000
Currently processing: 19/43
1000
Currently processing: 20/43
1000
Currently processing: 21/43
1000
Currently processing: 22/43
1000
Currently processing: 23/43
1000
Currently processing: 24/43
1000
Currently processing: 25/43
1000
Currently processing: 26/43
1000
Currently processing: 27/43
1000
Currently processing: 28/43
1000
Currently processing: 29/43
1000
Currently processing: 30/43
1000
Currently processin

In [9]:
result

,name,AP(rel=2),RR(rel=2),P(rel=2)@3,R(rel=2)@1000,nDCG@10,AP(rel=2) +,AP(rel=2) -,AP(rel=2) p-value,RR(rel=2) +,RR(rel=2) -,RR(rel=2) p-value,P(rel=2)@3 +,P(rel=2)@3 -,P(rel=2)@3 p-value,R(rel=2)@1000 +,R(rel=2)@1000 -,R(rel=2)@1000 p-value,nDCG@10 +,nDCG@10 -,nDCG@10 p-value
0,bm25,0.286448,0.641565,0.472868,0.755332,0.479540,8.000000,34.000000,0.000006,6.000000,19.000000,0.021567,5.000000,21.000000,0.000739,0.000000,0.000000,nan,8.000000,35.000000,0.000002
1,+GCN,0.425299,0.813437,0.689922,0.755332,0.674034,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
2,+GraphSAGE,0.414521,0.812108,0.689922,0.755332,0.668925,9.000000,29.000000,0.002204,3.000000,2.000000,0.921418,2.000000,2.000000,1.000000,0.000000,0.000000,nan,16.000000,18.000000,0.431621
3,+GAT,0.414469,0.815116,0.705426,0.755332,0.677581,14.000000,25.000000,0.022344,5.000000,2.000000,0.901914,3.000000,1.000000,0.323037,0.000000,0.000000,nan,16.000000,19.000000,0.648477
4,+GIN,0.413351,0.782521,0.674419,0.755332,0.664906,9.000000,30.000000,0.000175,1.000000,6.000000,0.081221,2.000000,4.000000,0.420647,0.000000,0.000000,nan,10.000000,24.000000,0.082158
5,+SignedConv,0.412732,0.820930,0.689922,0.755332,0.678174,9.000000,30.000000,0.006129,3.000000,2.000000,0.707587,2.000000,2.000000,1.000000,0.000000,0.000000,nan,15.000000,21.000000,0.603364


In [11]:
path = f'results/{embedding_name}/{modal}/trec-dl19/result_perquery_ablation.csv'
result_df = result.data
result_df.to_csv(path, sep = ' ', index = False)

In [12]:
dataset = pt.get_dataset('irds:msmarco-passage/trec-dl-2020/judged')
dl2020_dataset = dataset

filtered_get_topics = dataset.get_topics()
filtered_get_qrels = dataset.get_qrels()

result = pt.Experiment(
  [ 
    bm25,
    #bm25 >> pt.text.get_text(pt.get_dataset('irds:msmarco-passage'), 'text') >> TCTC,
    bm25 >> GNRR(tct_scorer_local_gcn, graph, flex_index, text_field=text_field), 
    bm25 >> GNRR(tct_scorer_local_sage, graph, flex_index, text_field=text_field), 
    bm25 >> GNRR(tct_scorer_local_gat, graph, flex_index, text_field=text_field), 
    bm25 >> GNRR(tct_scorer_local_gin, graph, flex_index, text_field=text_field), 
    bm25 >> GNRR(tct_scorer_local_signed, graph, flex_index, text_field=text_field) 


  ],
  filtered_get_topics,
  filtered_get_qrels,
  baseline = 1,
  test = test,
  highlight = 'bold',
  names=['bm25', ' +GCN', ' +GraphSAGE',' +GAT', ' +GIN', ' +SignedConv'],#, 'MLP', 'MLP + GCN', 'MLP + GIN'],#, 'MLP', 'monoT5'],#, "TCTColbert+GCN", "TCTColbert+MLP+GCN", "TCTColbert+MLP", "MonoT5"],
  eval_metrics=[nDCG@10, P(rel = 2)@3, AP(rel=2), RR(rel = 2), R(rel=2)@1000]
)
print(result)
# result = pt.Experiment(
#   [ 
#     bm25,
#     bm25 >> pt.text.get_text(pt.get_dataset('irds:msmarco-passage'), 'text') >> TCTC, 
#     bm25 >> GNRR(tct_scorer_global_gcn_sl, graph, flex_index, text_field=text_field),
#     bm25 >> GNRR(tct_scorer_global_gat_sl, graph, flex_index, text_field=text_field)

#   ],
#   filtered_get_topics,
#   filtered_get_qrels,
#   baseline = 1,
#   names=['bm25', 'TCTColbert', ' +GCN_Global_SL', ' +GAT_Global_SL'],
#   test = 't',
#   highlight = 'bold',#, 'MLP', 'MLP + GCN', 'MLP + GIN'],#, 'MLP', 'monoT5'],#, "TCTColbert+GCN", "TCTColbert+MLP+GCN", "TCTColbert+MLP", "MonoT5"],
#   eval_metrics=[nDCG@10, P(rel = 2)@3, AP(rel=2), RR(rel = 2), R(rel=2)@1000] #[nDCG@10, nDCG@20, P(rel = 2)@10, P(rel = 2)@20, R(rel=2)@1000]
# )

Currently processing: 1/54
1000
Currently processing: 2/54
1000
Currently processing: 3/54
1000
Currently processing: 4/54
1000
Currently processing: 5/54
1000
Currently processing: 6/54
1000
Currently processing: 7/54
1000
Currently processing: 8/54
1000
Currently processing: 9/54
1000
Currently processing: 10/54
1000
Currently processing: 11/54
1000
Currently processing: 12/54
1000
Currently processing: 13/54
1000
Currently processing: 14/54
1000
Currently processing: 15/54
1000
Currently processing: 16/54
1000
Currently processing: 17/54
1000
Currently processing: 18/54
1000
Currently processing: 19/54
1000
Currently processing: 20/54
1000
Currently processing: 21/54
1000
Currently processing: 22/54
1000
Currently processing: 23/54
1000
Currently processing: 24/54
1000
Currently processing: 25/54
1000
Currently processing: 26/54
1000
Currently processing: 27/54
1000
Currently processing: 28/54
1000
Currently processing: 29/54
1000
Currently processing: 30/54
1000
Currently processin

In [13]:
result

,name,AP(rel=2),RR(rel=2),P(rel=2)@3,R(rel=2)@1000,nDCG@10,AP(rel=2) +,AP(rel=2) -,AP(rel=2) p-value,RR(rel=2) +,RR(rel=2) -,RR(rel=2) p-value,P(rel=2)@3 +,P(rel=2)@3 -,P(rel=2)@3 p-value,R(rel=2)@1000 +,R(rel=2)@1000 -,R(rel=2)@1000 p-value,nDCG@10 +,nDCG@10 -,nDCG@10 p-value
0,bm25,0.292988,0.618666,0.462963,0.807223,0.493627,10.000000,44.000000,0.000001,6.000000,25.000000,0.000670,6.000000,30.000000,0.000085,0.000000,0.000000,nan,13.000000,41.000000,0.000012
1,+GCN,0.439466,0.813956,0.672840,0.807223,0.667072,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
2,+GraphSAGE,0.437083,0.798166,0.679012,0.807223,0.659700,23.000000,27.000000,0.609131,5.000000,5.000000,0.541008,3.000000,4.000000,0.798976,0.000000,0.000000,nan,17.000000,25.000000,0.290405
3,+GAT,0.437356,0.813786,0.679012,0.807223,0.661865,20.000000,30.000000,0.722896,8.000000,5.000000,0.995099,3.000000,3.000000,0.742271,0.000000,0.000000,nan,23.000000,24.000000,0.422750
4,+GIN,0.431057,0.812402,0.666667,0.807223,0.652040,24.000000,27.000000,0.052957,6.000000,6.000000,0.954149,2.000000,3.000000,0.658939,0.000000,0.000000,nan,17.000000,29.000000,0.062346
5,+SignedConv,0.432405,0.810460,0.666667,0.807223,0.660666,22.000000,30.000000,0.143937,5.000000,4.000000,0.881659,3.000000,5.000000,0.766141,0.000000,0.000000,nan,27.000000,21.000000,0.374995


In [14]:
path = f'results/{embedding_name}/{modal}/trec-dl20/result_perquery_ablation.csv'
result_df = result.data
result_df.to_csv(path, sep = ' ', index = False)

In [15]:
dataset = pt.get_dataset('irds:msmarco-passage/trec-dl-hard')
dl_hard_dataset = dataset
filtered_get_topics = dataset.get_topics()
filtered_get_qrels = dataset.get_qrels()


result = pt.Experiment(
  [ 
    bm25,
    #bm25 >> pt.text.get_text(pt.get_dataset('irds:msmarco-passage'), 'text') >> TCTC,
    bm25 >> GNRR(tct_scorer_local_gcn, graph, flex_index, text_field=text_field), 
    bm25 >> GNRR(tct_scorer_local_sage, graph, flex_index, text_field=text_field), 
    bm25 >> GNRR(tct_scorer_local_gat, graph, flex_index, text_field=text_field), 
    bm25 >> GNRR(tct_scorer_local_gin, graph, flex_index, text_field=text_field), 
    bm25 >> GNRR(tct_scorer_local_signed, graph, flex_index, text_field=text_field) 


  ],
  filtered_get_topics,
  filtered_get_qrels,
  baseline = 1,
  test = test,
  highlight = 'bold',
  names=['bm25', ' +GCN', ' +GraphSAGE',' +GAT', ' +GIN', ' +SignedConv'],#, 'MLP', 'MLP + GCN', 'MLP + GIN'],#, 'MLP', 'monoT5'],#, "TCTColbert+GCN", "TCTColbert+MLP+GCN", "TCTColbert+MLP", "MonoT5"],
  eval_metrics=[nDCG@10, P(rel = 2)@3, AP(rel=2), RR(rel = 2), R(rel=2)@1000]
)
print(result)
# result = pt.Experiment(
#   [ 
#     bm25,
#     bm25 >> pt.text.get_text(pt.get_dataset('irds:msmarco-passage'), 'text') >> TCTC, 
#     bm25 >> GNRR(tct_scorer_global_gcn_sl, graph, flex_index, text_field=text_field),
#     bm25 >> GNRR(tct_scorer_global_gat_sl, graph, flex_index, text_field=text_field)

#   ],
#   filtered_get_topics,
#   filtered_get_qrels,
#   baseline = 1,
#   names=['bm25', 'TCTColbert', ' +GCN_Global_SL', ' +GAT_Global_SL'],
#   test = 't',
#   highlight = 'bold',#, 'MLP', 'MLP + GCN', 'MLP + GIN'],#, 'MLP', 'monoT5'],#, "TCTColbert+GCN", "TCTColbert+MLP+GCN", "TCTColbert+MLP", "MonoT5"],
#   eval_metrics=[nDCG@10, P(rel = 2)@3, AP(rel=2), RR(rel = 2), R(rel=2)@1000] #[nDCG@10, nDCG@20, P(rel = 2)@10, P(rel = 2)@20, R(rel=2)@1000]
# )

Currently processing: 1/50
1000
Currently processing: 2/50
1000
Currently processing: 3/50
1000
Currently processing: 4/50
1000
Currently processing: 5/50
1000
Currently processing: 6/50
1000
Currently processing: 7/50
1000
Currently processing: 8/50
1000
Currently processing: 9/50
1000
Currently processing: 10/50
1000
Currently processing: 11/50
1000
Currently processing: 12/50
1000
Currently processing: 13/50
1000
Currently processing: 14/50
1000
Currently processing: 15/50
1000
Currently processing: 16/50
1000
Currently processing: 17/50
1000
Currently processing: 18/50
1000
Currently processing: 19/50
1000
Currently processing: 20/50
1000
Currently processing: 21/50
1000
Currently processing: 22/50
1000
Currently processing: 23/50
1000
Currently processing: 24/50
1000
Currently processing: 25/50
1000
Currently processing: 26/50
1000
Currently processing: 27/50
1000
Currently processing: 28/50
1000
Currently processing: 29/50
1000
Currently processing: 30/50
1000
Currently processin

In [16]:
result

,name,AP(rel=2),RR(rel=2),P(rel=2)@3,R(rel=2)@1000,nDCG@10,AP(rel=2) +,AP(rel=2) -,AP(rel=2) p-value,RR(rel=2) +,RR(rel=2) -,RR(rel=2) p-value,P(rel=2)@3 +,P(rel=2)@3 -,P(rel=2)@3 p-value,R(rel=2)@1000 +,R(rel=2)@1000 -,R(rel=2)@1000 p-value,nDCG@10 +,nDCG@10 -,nDCG@10 p-value
0,bm25,0.147106,0.422003,0.240000,0.686557,0.274333,15.000000,32.000000,0.026381,12.000000,27.000000,0.160608,7.000000,21.000000,0.012768,0.000000,0.000000,nan,13.000000,29.000000,0.018779
1,+GCN,0.219015,0.519936,0.360000,0.686557,0.360433,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
2,+GraphSAGE,0.209546,0.480657,0.333333,0.686557,0.342576,20.000000,24.000000,0.204354,11.000000,13.000000,0.110605,0.000000,4.000000,0.044315,0.000000,0.000000,nan,15.000000,17.000000,0.154863
3,+GAT,0.211931,0.488434,0.340000,0.686557,0.353434,25.000000,20.000000,0.354655,13.000000,12.000000,0.193583,1.000000,4.000000,0.182352,0.000000,0.000000,nan,18.000000,18.000000,0.501340
4,+GIN,0.209996,0.500158,0.333333,0.686557,0.357991,19.000000,24.000000,0.097449,10.000000,13.000000,0.387464,2.000000,6.000000,0.159386,0.000000,0.000000,nan,16.000000,20.000000,0.822503
5,+SignedConv,0.209726,0.497186,0.340000,0.686557,0.354647,21.000000,23.000000,0.226689,11.000000,11.000000,0.333408,1.000000,4.000000,0.182352,0.000000,0.000000,nan,18.000000,17.000000,0.571608


In [18]:
path = f'results/{embedding_name}/{modal}/trec-dlhard/result_perquery_ablation.csv'
result_df = result.data
result_df.to_csv(path, sep = ' ', index = False)

In [10]:
dataset = pt.get_dataset('irds:msmarco-passage/dev/small')
dl19_2019 = dataset
filtered_get_topics = dataset.get_topics()[6000:]
filtered_get_qrels = dataset.get_qrels()[dataset.get_qrels()['qid'].isin(filtered_get_topics['qid'])]
from pyterrier.measures import * 


result = pt.Experiment(
  [ 
    bm25,
    bm25 >> pt.text.get_text(pt.get_dataset('irds:msmarco-passage'), 'text') >> TCTC, 
    bm25 >> GNRR(tct_scorer_local_gcn_multi, graph, flex_index, text_field=text_field),

  ],
  filtered_get_topics,
  filtered_get_qrels,
  baseline = 1,
  test = test,
  highlight = 'bold',
  names=['bm25', 'TCTColBert', ' +GCN'],#, ' +GraphSAGE',' +GAT', ' +GIN', ' +SignedConv'],#, 'MLP', 'MLP + GCN', 'MLP + GIN'],#, 'MLP', 'monoT5'],#, "TCTColbert+GCN", "TCTColbert+MLP+GCN", "TCTColbert+MLP", "MonoT5"],
  eval_metrics=[nDCG@10, P@3, AP, RR, R@1000]
)
print(result)
# result = pt.Experiment(
#   [ 
#     bm25,
#     bm25 >> pt.text.get_text(pt.get_dataset('irds:msmarco-passage'), 'text') >> TCTC, 
#     bm25 >> GNRR(tct_scorer_global_gcn_sl, graph, flex_index, text_field=text_field),
#     bm25 >> GNRR(tct_scorer_global_gat_sl, graph, flex_index, text_field=text_field)

#   ],
#   filtered_get_topics,
#   filtered_get_qrels,
#   baseline = 1,
#   names=['bm25', 'TCTColbert', ' +GCN_Global_SL', ' +GAT_Global_SL'],
#   test = 't',
#   highlight = 'bold',#, 'MLP', 'MLP + GCN', 'MLP + GIN'],#, 'MLP', 'monoT5'],#, "TCTColbert+GCN", "TCTColbert+MLP+GCN", "TCTColbert+MLP", "MonoT5"],
#   eval_metrics=[nDCG@10, P(rel = 2)@3, AP(rel=2), RR(rel = 2), R(rel=2)@1000] #[nDCG@10, nDCG@20, P(rel = 2)@10, P(rel = 2)@20, R(rel=2)@1000]
# )

Currently processing: 1/980
1000
Currently processing: 2/980
1000
Currently processing: 3/980
1000
Currently processing: 4/980
1000
Currently processing: 5/980
1000
Currently processing: 6/980
1000
Currently processing: 7/980
1000
Currently processing: 8/980
1000
Currently processing: 9/980
1000
Currently processing: 10/980
1000
Currently processing: 11/980
1000
Currently processing: 12/980
1000
Currently processing: 13/980
1000
Currently processing: 14/980
1000
Currently processing: 15/980
1000
Currently processing: 16/980
1000
Currently processing: 17/980
1000
Currently processing: 18/980
1000
Currently processing: 19/980
1000
Currently processing: 20/980
1000
Currently processing: 21/980
1000
Currently processing: 22/980
1000
Currently processing: 23/980
1000
Currently processing: 24/980
1000
Currently processing: 25/980
1000
Currently processing: 26/980
1000
Currently processing: 27/980
1000
Currently processing: 28/980
1000
Currently processing: 29/980
1000
Currently processing: 3

In [11]:
result

,name,AP,RR,P@3,R@1000,nDCG@10,AP +,AP -,AP p-value,RR +,RR -,RR p-value,P@3 +,P@3 -,P@3 p-value,R@1000 +,R@1000 -,R@1000 p-value,nDCG@10 +,nDCG@10 -,nDCG@10 p-value
0,bm25,0.188466,0.191342,0.075170,0.869643,0.224926,185.000000,598.000000,0.000000,185.000000,594.000000,0.000000,55.000000,247.000000,0.000000,0.000000,0.000000,nan,120.000000,433.000000,0.000000
1,TCTColBert,0.350862,0.353837,0.142177,0.869643,0.402557,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
2,+GCN,0.338727,0.341963,0.137755,0.869643,0.391565,189.000000,228.000000,0.009197,187.000000,224.000000,0.012919,22.000000,34.000000,0.090570,0.000000,0.000000,nan,105.000000,137.000000,0.006881


In [ ]:
path = f'results/{embedding_name}/{modal}/devsmall/result.csv'
result_df = result#.data
result_df.to_csv(path, sep = ' ', index = False)

## Load results


In [ ]:
import pandas as pd
embedding_name = 'tctcolbert'
modal = 'local'

In [ ]:
def round_float_values(df):
    for column in df.columns:
        if pd.api.types.is_float_dtype(df[column]):
            df[column] = df[column].round(4)
    return df

def remove_plus_minus_columns(df):
    for column in df.columns:

        if '+' in column or '-' in column or column.startswith('R(rel=2)@'):  # if column name contains '+' or '-'
            
            if column[-7:] == 'p-value' and not column.startswith('R(rel=2)@'):
                continue
            df = df.drop(column, axis=1)  # drop the column
    return df

In [ ]:
path = f'results/{embedding_name}/{modal}/trec-dl19/dl19_result_perquery.csv'
result_df = pd.read_csv(path, sep = ' ')
# result_df = round_float_values(result_df)

result_df

In [ ]:
path = f'results/{embedding_name}/{modal}/trec-dl20/result.csv'
result_df = pd.read_csv(path, sep = ' ')
result_df = round_float_values(result_df)
result_df

In [ ]:
path = f'results/{embedding_name}/{modal}/trec-dlhard/result.csv'
result_df = pd.read_csv(path, sep = ' ')
result_df = round_float_values(result_df)
result_df

In [ ]:
dataset_name = "irds:msmarco-passage/train/split200-train"

train_dataset = pt.get_dataset(dataset_name)

dataset_name = "irds:msmarco-passage/train/split200-valid"

val_dataset = pt.get_dataset(dataset_name)

dataset_name = "irds:msmarco-passage/dev/small"

dev_dataset = pt.get_dataset(dataset_name)

dataset_name = "irds:msmarco-passage/trec-dl-2019/judged"

dl19_dataset = pt.get_dataset(dataset_name)


dataset_name = "irds:msmarco-passage/trec-dl-2020/judged"

dl20_dataset = pt.get_dataset(dataset_name)


dataset_name = "irds:msmarco-passage/trec-dl-hard"

dlhard_dataset = pt.get_dataset(dataset_name)

In [ ]:
set_query_train = set(train_dataset.get_topics()['query'][:15000])
set_query_val = set(val_dataset.get_topics()['query'])
# set_query_dl19 = set(dl19_dataset.get_topics()['query'])
# set_query_dl20 = set(dl20_dataset.get_topics()['query'])
# set_query_dl_hard = set(dlhard_dataset.get_topics()['query'])
set_query_dev = set(dev_dataset.get_topics()['query'])



In [ ]:
len(set_query_train.intersection(set_query_val))

In [ ]:
# dataset = pt.get_dataset('irds:msmarco-passage/train/split200-valid')

# bm25 = pt.BatchRetrieve.from_dataset('msmarco_passage', 'terrier_stemmed', wmodel='BM25')

# pipeline = bm25 #>> pt.text.get_text(pt.get_dataset('irds:msmarco-passage'), 'text')
# data_query = dataset.get_topics()

# for i in range(0, len(data_query)):
#     print(f"Currently processing query {i+1}/{len(data_query)}")
#     query_id = data_query.loc[i].qid
#     path_to_save = f'data/msmarco_data/msmarco_pre-computed_bm25/val/{data_query.loc[i].qid}.json'
    
#     if os.path.exists(path_to_save):
#         print("skipped")
#         continue

#     output = pipeline(data_query.iloc[i:i+1, :])
    
#     with open(path_to_save, 'w') as f:
#         json.dump(output.to_dict(), f)